# BDT, Deep Sets, and MLP performance comparison

This notebook compares the three proton-free regressors using plots modeled on Kuttan *et al.*, [*A fast centrality-meter for heavy-ion collisions at the CBM experiment*](https://arxiv.org/abs/2009.01584). The paper reports mean prediction error, relative precision $\sigma_{\mathrm{err}}/y_{\mathrm{true}}$, and mean error in 5% centrality classes.

Here the same diagnostics are shown for both impact parameter $b$ and participant count $N_{\mathrm{part}}$, followed by an overall MAE bar chart. Error is defined throughout as $y_{\mathrm{true}}-y_{\mathrm{pred}}$.

## 1. Imports and result locations

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np

search_roots = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (root for root in search_roots if (root / "Glauber/nuclei.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the Deep_Learning_Npart project root.")

RESULTS_DIR = PROJECT_ROOT / "results/model_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILES = {
    "Boosted tree": PROJECT_ROOT / "results/boosted_tree/boosted_tree_results.npz",
    "Deep Sets": PROJECT_ROOT / "results/deepset/deepset_results.npz",
    "MLP": PROJECT_ROOT / "results/perceptron/perceptron_results.npz",
}
COLORS = {
    "Boosted tree": "tab:blue",
    "Deep Sets": "tab:orange",
    "MLP": "tab:green",
}
MARKERS = {"Boosted tree": "o", "Deep Sets": "s", "MLP": "^"}
MIN_BIN_COUNT = 10

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})
print("Comparison results:", RESULTS_DIR)

## 2. Load and validate held-out predictions

Run the BDT, DeepSet, and MLP notebooks first so that each writes its comparison artifact. If dataset identifiers or test-event IDs differ, the notebook issues a warning and computes every curve on that model's own held-out sample. For a strict ranking, all models should use the same event sample and split.

In [ ]:
REQUIRED_KEYS = (
    "model_name", "dataset_id", "test_event_id",
    "true_Npart", "true_b", "pred_Npart", "pred_b",
    "centrality_signal",
)


def load_model_result(path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Run its model notebook through the final save cell first."
        )
    with np.load(path, allow_pickle=False) as data:
        missing = [key for key in REQUIRED_KEYS if key not in data.files]
        if missing:
            raise KeyError(f"{path} is missing comparison keys: {missing}")
        result = {
            "model_name": str(data["model_name"].item()),
            "dataset_id": str(data["dataset_id"].item()),
            "event_id": data["test_event_id"].astype(np.int64),
            "true_Npart": data["true_Npart"].astype(float),
            "true_b": data["true_b"].astype(float),
            "pred_Npart": data["pred_Npart"].astype(float),
            "pred_b": data["pred_b"].astype(float),
            "centrality_signal": data["centrality_signal"].astype(float),
        }

    lengths = {len(value) for key, value in result.items() if isinstance(value, np.ndarray)}
    if len(lengths) != 1:
        raise ValueError(f"Inconsistent array lengths in {path}: {sorted(lengths)}")
    numeric_arrays = [value for value in result.values() if isinstance(value, np.ndarray)]
    if not all(np.all(np.isfinite(value)) for value in numeric_arrays):
        raise ValueError(f"Non-finite values found in {path}")
    return result


models = {label: load_model_result(path) for label, path in MODEL_FILES.items()}

first = next(iter(models.values()))
same_dataset = all(model["dataset_id"] == first["dataset_id"] for model in models.values())
same_events = same_dataset and all(
    np.array_equal(model["event_id"], first["event_id"]) for model in models.values()
)
same_truth = same_events and all(
    np.allclose(model["true_b"], first["true_b"])
    and np.allclose(model["true_Npart"], first["true_Npart"])
    for model in models.values()
)
same_evaluation = same_events and same_truth
if not same_evaluation:
    warnings.warn(
        "Models were not evaluated on an identical dataset/event order. "
        "The plots remain descriptive, but overall MAE is not a strict apples-to-apples ranking.",
        stacklevel=1,
    )

for label, model in models.items():
    print(f"{label:13s}  n={len(model['event_id']):6,d}  dataset={model['dataset_id']}")
print("Identical evaluation events and truth values:", same_evaluation)

## 3. Shared statistics and plotting helpers

In [ ]:
def binned_error_statistics(truth, prediction, edges, min_count=MIN_BIN_COUNT):
    truth = np.asarray(truth, dtype=float)
    error = truth - np.asarray(prediction, dtype=float)
    bin_index = np.digitize(truth, edges) - 1
    centers = 0.5 * (edges[:-1] + edges[1:])
    mean_error = np.full(len(centers), np.nan)
    error_sem = np.full(len(centers), np.nan)
    relative_precision = np.full(len(centers), np.nan)
    counts = np.zeros(len(centers), dtype=int)

    for index in range(len(centers)):
        selected = bin_index == index
        counts[index] = selected.sum()
        if counts[index] < min_count:
            continue
        bin_error = error[selected]
        denominator = np.abs(truth[selected].mean())
        mean_error[index] = bin_error.mean()
        error_sem[index] = bin_error.std(ddof=1) / np.sqrt(counts[index])
        if denominator > 1e-12:
            relative_precision[index] = bin_error.std(ddof=1) / denominator

    return {
        "centers": centers,
        "mean_error": mean_error,
        "error_sem": error_sem,
        "relative_precision": relative_precision,
        "counts": counts,
    }


def centrality_error_statistics(model, target_key, prediction_key):
    signal = model["centrality_signal"]
    order = np.argsort(-signal, kind="stable")
    percentile = np.empty(len(signal), dtype=float)
    percentile[order] = 100.0 * (np.arange(len(signal)) + 0.5) / len(signal)
    edges = np.arange(0.0, 105.0, 5.0)
    return _centrality_stats(percentile, model[target_key] - model[prediction_key], edges)


def _centrality_stats(percentile, error, edges):
    bin_index = np.digitize(percentile, edges) - 1
    centers = 0.5 * (edges[:-1] + edges[1:])
    means = np.full(len(centers), np.nan)
    sems = np.full(len(centers), np.nan)
    counts = np.zeros(len(centers), dtype=int)
    for index in range(len(centers)):
        selected = bin_index == index
        counts[index] = selected.sum()
        if counts[index] >= MIN_BIN_COUNT:
            means[index] = error[selected].mean()
            sems[index] = error[selected].std(ddof=1) / np.sqrt(counts[index])
    return {"centers": centers, "mean_error": means, "error_sem": sems, "counts": counts}


def save_figure(fig, stem):
    for suffix in ("png", "pdf"):
        fig.savefig(RESULTS_DIR / f"{stem}.{suffix}", dpi=220, bbox_inches="tight")

## 4. CBM-style mean error versus true geometry

As in the CBM paper, the vertical coordinate is the mean of the event-level error distribution in each truth bin. Error bars show the standard error of that mean.

In [ ]:
max_b = max(np.max(model["true_b"]) for model in models.values())
max_npart = max(np.max(model["true_Npart"]) for model in models.values())
b_edges = np.arange(0.0, np.ceil(max_b) + 1.0, 1.0)
npart_upper = (np.floor(max_npart / 20.0) + 1.0) * 20.0
npart_edges = np.linspace(0.0, npart_upper, 21)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, model in models.items():
    for ax, truth_key, pred_key, edges in (
        (axes[0], "true_b", "pred_b", b_edges),
        (axes[1], "true_Npart", "pred_Npart", npart_edges),
    ):
        stats = binned_error_statistics(model[truth_key], model[pred_key], edges)
        valid = np.isfinite(stats["mean_error"])
        ax.errorbar(
            stats["centers"][valid],
            stats["mean_error"][valid],
            yerr=stats["error_sem"][valid],
            marker=MARKERS[label],
            markersize=4,
            capsize=2,
            color=COLORS[label],
            label=label,
        )

axes[0].set(xlabel=r"True $b$ [fm]", ylabel=r"Mean error $b_{\rm true}-b_{\rm pred}$ [fm]")
axes[1].set(xlabel=r"True $N_{\rm part}$", ylabel=r"Mean error $N_{\rm part}^{true}-N_{\rm part}^{pred}$")
for ax in axes:
    ax.axhline(0.0, color="black", linewidth=1)
    ax.legend()
fig.suptitle("CBM-style accuracy: mean prediction error")
fig.tight_layout()
save_figure(fig, "mean_error_vs_truth_cbm_style")
plt.show()

## 5. CBM-style relative precision

Relative precision is the standard deviation of $y_{\mathrm{true}}-y_{\mathrm{pred}}$ divided by the mean truth value in the bin. It measures spread, so it should be interpreted together with the mean-error plot.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, model in models.items():
    for ax, truth_key, pred_key, edges in (
        (axes[0], "true_b", "pred_b", b_edges),
        (axes[1], "true_Npart", "pred_Npart", npart_edges),
    ):
        stats = binned_error_statistics(model[truth_key], model[pred_key], edges)
        valid = np.isfinite(stats["relative_precision"])
        ax.plot(
            stats["centers"][valid],
            100.0 * stats["relative_precision"][valid],
            marker=MARKERS[label],
            markersize=4,
            color=COLORS[label],
            label=label,
        )

axes[0].set(xlabel=r"True $b$ [fm]", ylabel=r"Relative precision $\sigma_{err}/b_{true}$ [%]")
axes[1].set(xlabel=r"True $N_{\rm part}$", ylabel=r"Relative precision $\sigma_{err}/N_{\rm part}^{true}$ [%]")
for ax in axes:
    ax.legend()
fig.suptitle("CBM-style precision versus true geometry")
fig.tight_layout()
save_figure(fig, "relative_precision_cbm_style")
plt.show()

## 6. Mean error versus 5% centrality class

Events are ranked from largest to smallest saved multiplicity signal, matching the paper's multiplicity-based centrality convention.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for label, model in models.items():
    for ax, truth_key, pred_key in (
        (axes[0], "true_b", "pred_b"),
        (axes[1], "true_Npart", "pred_Npart"),
    ):
        stats = centrality_error_statistics(model, truth_key, pred_key)
        valid = np.isfinite(stats["mean_error"])
        ax.errorbar(
            stats["centers"][valid],
            stats["mean_error"][valid],
            yerr=stats["error_sem"][valid],
            marker=MARKERS[label],
            markersize=4,
            capsize=2,
            color=COLORS[label],
            label=label,
        )

axes[0].set(xlabel="Centrality [%]", ylabel=r"Mean error $b_{\rm true}-b_{\rm pred}$ [fm]")
axes[1].set(xlabel="Centrality [%]", ylabel=r"Mean error $N_{\rm part}^{true}-N_{\rm part}^{pred}$")
for ax in axes:
    ax.axhline(0.0, color="black", linewidth=1)
    ax.set_xlim(0, 100)
    ax.legend()
fig.suptitle("CBM-style mean error by multiplicity centrality")
fig.tight_layout()
save_figure(fig, "mean_error_vs_centrality_cbm_style")
plt.show()

## 7. Overall MAE for $b$ and $N_{\mathrm{part}}$

In [ ]:
labels = list(models)
mae_b = [np.mean(np.abs(models[label]["true_b"] - models[label]["pred_b"])) for label in labels]
mae_npart = [
    np.mean(np.abs(models[label]["true_Npart"] - models[label]["pred_Npart"]))
    for label in labels
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
for ax, values, ylabel, title, precision in (
    (axes[0], mae_b, r"MAE in $b$ [fm]", r"Overall impact-parameter MAE", 3),
    (axes[1], mae_npart, r"MAE in $N_{\mathrm{part}}$", r"Overall participant-count MAE", 2),
):
    bars = ax.bar(labels, values, color=[COLORS[label] for label in labels])
    ax.set(ylabel=ylabel, title=title)
    ax.grid(axis="y", alpha=0.25)
    ax.grid(axis="x", visible=False)
    ax.bar_label(bars, fmt=f"%.{precision}f", padding=3)
    ax.set_ylim(0, max(values) * 1.18)

fig.tight_layout()
save_figure(fig, "overall_mae_models")
plt.show()

print(f"{'Model':13s}  {'MAE b [fm]':>12s}  {'MAE Npart':>12s}")
for label, b_value, npart_value in zip(labels, mae_b, mae_npart):
    print(f"{label:13s}  {b_value:12.4f}  {npart_value:12.4f}")

## Output files

Each figure is saved as both PNG and PDF under `results/model_comparison`:

- `mean_error_vs_truth_cbm_style`
- `relative_precision_cbm_style`
- `mean_error_vs_centrality_cbm_style`
- `overall_mae_models`